# Province classification study - 4 approaches, same data, same test set

Which architecture / training strategy reads the Khmer province line best?
Every run uses the identical split (2,515 / 529 / 567), the identical
augmentation, the identical test-time preprocessing, 40 epochs, seed 42.

| run | architecture | initial weights | what trains | rubric dimension |
|---|---|---|---|---|
| **A** | ResNet18 | random | everything | training strategy |
| **B** | ResNet18 | ImageNet | **only the final layer** (feature extraction / linear probe) | training strategy |
| **C** | ResNet18 | ImageNet | everything, low LR (fine-tuning) | training strategy |
| **D** | small 4-block CNN (0.4 M params) | random | everything | architecture |

Each run writes `results/province_study/<run>/` with `history.csv` (per-epoch
train/val loss + accuracy), `run.json` (params, time, GPU, test accuracy,
macro-F1) and `test_predictions.csv` (every test image with true / predicted
class). The last cell copies that folder to Drive - commit it to the repo.

Run time: roughly 10 min per ResNet18 run on a T4, less for D.


In [1]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-52bd96d7-33ce-55d6-6dce-4e918c6af837)


In [2]:
from google.colab import drive
drive.mount('/content/drive')
BUNDLE = '/content/drive/MyDrive/ALPR/alpr_colab_bundle.zip'

import os, zipfile, shutil
shutil.rmtree('/content/alpr', ignore_errors=True)
os.makedirs('/content/alpr', exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    z.extractall('/content/alpr')
%cd /content/alpr

Mounted at /content/drive
/content/alpr


In [3]:
# ---- BUNDLE FRESHNESS CHECK -- do not skip -------------------------------
import glob

tr = open('scripts/recognition/train_province_classifier.py', encoding='utf-8').read()
needed = ['--arch', '--weight-decay', '--seed', '--resume', 'history.csv', 'run.json']
missing = [f for f in needed if f not in tr]
n_train = len(glob.glob('data/province_crops/train/*/*.jpg'))
n_test  = len(glob.glob('data/province_crops/test/*/*.jpg'))

print('missing flags   :', missing or 'none')
print('province crops  : train', n_train, '| test', n_test)
assert not missing, 'STALE BUNDLE -- run make_colab_bundle.py again and re-upload.'
assert n_train >= 2500 and n_test >= 560, 'province crops missing'
print()
print('Bundle is current. Safe to continue.')

missing flags   : none
province crops  : train 2515 | test 567

Bundle is current. Safe to continue.


In [4]:
!pip -q install torch torchvision pyyaml tqdm pillow opencv-python-headless numpy

In [5]:
# ---- RUN A: ResNet18 from scratch ----------------------------------------
!python scripts/recognition/train_province_classifier.py --epochs 40 --batch 32 --seed 42 \
    --arch resnet18 --lr 1e-3 \
    --run-name A_resnet18_scratch --out models/recognition/prov_A_scratch.pth

 TRAIN PROVINCE CLASSIFIER (resnet18, 26 classes)
   device cuda | epochs 40 | batch 32
   output prov_A_scratch.pth | mode=from-scratch | framing_aug=True | rotate=False | rotate180=False | seed=42
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
train=2515 val=529 test=567 | classes=26 | idx_to_class=[0, 1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 2, 20, 21, 22, 23, 24, 25, 3, 4, 5, 6, 7, 8, 9]
[freeze] FINE-TUNING: all 11,189,850 parameters trainable
   history -> results/province_study/A_resnet18_scratch/history.csv
Epoch 1/40:   0% 0/79 [00:00<?, ?it/s]/usr/lo

In [6]:
# ---- RUN B: ResNet18 feature extraction (ImageNet frozen, head only) -----
# Watch the [freeze] line: ~11.18 M frozen, 13,338 trainable (0.12%).
!python scripts/recognition/train_province_classifier.py --epochs 40 --batch 32 --seed 42 \
    --arch resnet18 --pretrained --freeze --lr 1e-3 \
    --run-name B_resnet18_frozen --out models/recognition/prov_B_featext.pth

 TRAIN PROVINCE CLASSIFIER (resnet18, 26 classes)
   device cuda | epochs 40 | batch 32
   output prov_B_featext.pth | mode=feature-extraction | framing_aug=True | rotate=False | rotate180=False | seed=42
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
train=2515 val=529 test=567 | classes=26 | idx_to_class=[0, 1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 2, 20, 21, 22, 23, 24, 25, 3, 4, 5, 6, 7, 8, 9]
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 1

In [7]:
# ---- RUN C: ResNet18 fine-tuning (ImageNet, all layers, low LR) ----------
!python scripts/recognition/train_province_classifier.py --epochs 40 --batch 32 --seed 42 \
    --arch resnet18 --pretrained --lr 1e-4 \
    --run-name C_resnet18_finetune --out models/recognition/prov_C_finetune.pth

 TRAIN PROVINCE CLASSIFIER (resnet18, 26 classes)
   device cuda | epochs 40 | batch 32
   output prov_C_finetune.pth | mode=fine-tuning | framing_aug=True | rotate=False | rotate180=False | seed=42
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
train=2515 val=529 test=567 | classes=26 | idx_to_class=[0, 1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 2, 20, 21, 22, 23, 24, 25, 3, 4, 5, 6, 7, 8, 9]
[freeze] FINE-TUNING: all 11,189,850 parameters trainable
   history -> results/province_study/C_resnet18_finetune/history.csv
Epoch 1/40:   0% 0/79 [00:00<?, ?it/s]/usr/l

In [8]:
# ---- RUN D: small CNN from scratch (low-capacity baseline) ---------------
!python scripts/recognition/train_province_classifier.py --epochs 40 --batch 32 --seed 42 \
    --arch small_cnn --lr 1e-3 \
    --run-name D_smallcnn_scratch --out models/recognition/prov_D_smallcnn.pth

 TRAIN PROVINCE CLASSIFIER (small_cnn, 26 classes)
   device cuda | epochs 40 | batch 32
   output prov_D_smallcnn.pth | mode=from-scratch | framing_aug=True | rotate=False | rotate180=False | seed=42
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
train=2515 val=529 test=567 | classes=26 | idx_to_class=[0, 1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 2, 20, 21, 22, 23, 24, 25, 3, 4, 5, 6, 7, 8, 9]
[freeze] FINE-TUNING: all 396,058 parameters trainable
   history -> results/province_study/D_smallcnn_scratch/history.csv
Epoch 1/40:   0% 0/79 [00:00<?, ?it/s]/usr/loc

In [9]:
# ---- RESULTS TABLE (from run.json) ---------------------------------------
import json, glob
rows = []
for p in sorted(glob.glob('results/province_study/*/run.json')):
    r = json.load(open(p))
    rows.append(r)
print('%-22s %-10s %-9s %12s %8s %9s %9s %8s' % (
    'run', 'arch', 'mode', 'trainable', 'best_ep', 'val_acc', 'test_acc', 'macroF1'))
print('-' * 96)
for r in rows:
    print('%-22s %-10s %-9s %12s %8d %8.2f%% %8.2f%% %7.2f%%  (%.1f min)' % (
        r['run_name'], r['arch'], r['mode'][:9], f"{r['trainable_params']:,}",
        r['best_epoch'], 100*r['best_val_acc'], 100*r['test_acc'],
        100*r['test_macro_f1'], r['train_wall_sec']/60))
print()
print('Deployed model for reference: 96.1% upright on the same 567 crops (test_province_rotation.py).')

run                    arch       mode         trainable  best_ep   val_acc  test_acc  macroF1
------------------------------------------------------------------------------------------------
A_resnet18_scratch     resnet18   from-scra   11,189,850       38    93.76%    93.30%   92.92%  (6.1 min)
B_resnet18_frozen      resnet18   feature-e       13,338       39    40.08%    41.98%   37.85%  (5.3 min)
C_resnet18_finetune    resnet18   fine-tuni   11,189,850       31    92.63%    94.00%   92.44%  (5.9 min)
D_smallcnn_scratch     small_cnn  from-scra      396,058       33    48.39%    46.56%   40.74%  (5.3 min)

Deployed model for reference: 96.1% upright on the same 567 crops (test_province_rotation.py).


### How to read it

* **A vs C** - does ImageNet initialisation help when the whole network trains? If they tie, 2.5 k images are enough to learn from scratch and pretraining only speeds convergence (check `best_epoch`).
* **B** - a frozen ImageNet backbone is a fixed feature extractor. If it collapses, ImageNet features do not separate Khmer glyphs: the low-level filters must be re-learned (domain shift / inductive bias).
* **D vs A** - capacity: does a 0.4 M-param CNN keep up with an 11 M-param ResNet at 128 px?
* Open `history.csv` for each run to see over-fitting (train >> val) or under-fitting (both flat).


In [10]:
# ---- SAVE EVERYTHING TO DRIVE ---------------------------------------------
# results/province_study/  -> commit this folder to the repo (CSV/JSON only)
# models/recognition/prov_*  -> weights (35-45 MB each) for Drive links
import shutil, os, glob
dst = '/content/drive/MyDrive/ALPR/province_study'
shutil.rmtree(dst, ignore_errors=True)
shutil.copytree('results/province_study', dst,
                ignore=shutil.ignore_patterns('last.pth'))
os.makedirs('/content/drive/MyDrive/ALPR/trained', exist_ok=True)
for f in glob.glob('models/recognition/prov_*'):
    shutil.copy(f, '/content/drive/MyDrive/ALPR/trained/')
print('saved ->', dst)
print('weights ->', sorted(os.listdir('/content/drive/MyDrive/ALPR/trained')))

saved -> /content/drive/MyDrive/ALPR/province_study
weights -> ['crnn_stn5.pth', 'prov_A_scratch.pth', 'prov_A_scratch_config.json', 'prov_B_featext.pth', 'prov_B_featext_config.json', 'prov_C_finetune.pth', 'prov_C_finetune_config.json', 'prov_D_smallcnn.pth', 'prov_D_smallcnn_config.json']
